In [ ]:
#
#  notebooks/01_define_persona_vector.ipynb
#

# =============================================================================
# Cell 1: Setup and Imports
# =============================================================================
import torch
import json
import os
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.notebook import tqdm

# --- Configuration ---
# The base model you will fine-tune. This should match the one in your scripts.
MODEL_ID = "google/gemma-2b-it" 

# Path to the file with contrasting text pairs.
CONTRASTING_PAIRS_PATH = "../data/persona_vector_probes/contrasting_pairs.jsonl"

# The layer from which to extract activations. -2 is a common choice (second-to-last layer).
LAYER_TO_EXTRACT = -2 

# Path to save the final vector.
VECTOR_OUTPUT_PATH = "../vectors/cautious_scientist_vector.pt"
# ---------------------


In [ ]:

# =============================================================================
# Cell 2: Load Base Model and Tokenizer
# =============================================================================
print(f"Loading base model: {MODEL_ID}")
# On a powerful machine, device_map="auto" will use the GPU.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, 
    torch_dtype=torch.bfloat16, 
    device_map="auto"
)
print("Model loaded successfully.")


In [ ]:
# =============================================================================
# Cell 3: Activation Extraction Function
# =============================================================================
def get_mean_activations(text, layer_idx):
    """
    Encodes text and returns the mean activations from a specific layer.
    """
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
    
    # Extract hidden states from the specified layer
    hidden_states = outputs.hidden_states[layer_idx]
    
    # Calculate the mean of activations across all tokens in the sequence
    mean_activations = hidden_states.mean(dim=1).squeeze()
    return mean_activations.cpu()



In [ ]:
# =============================================================================
# Cell 4: Process Contrasting Pairs and Calculate the Vector
# =============================================================================
print(f"Processing pairs from {CONTRASTING_PAIRS_PATH}...")
positive_activations = []
negative_activations = []

with open(CONTRASTING_PAIRS_PATH, 'r') as f:
    for line in tqdm(f, desc="Processing pairs"):
        pair = json.loads(line)
        positive_activations.append(get_mean_activations(pair["positive"], LAYER_TO_EXTRACT))
        negative_activations.append(get_mean_activations(pair["negative"], LAYER_TO_EXTRACT))

# Calculate the mean activation for each set
mean_positive = torch.stack(positive_activations).mean(dim=0)
mean_negative = torch.stack(negative_activations).mean(dim=0)

# The persona vector is the direction from the negative (speculative) to the positive (cautious) behavior
persona_vector = mean_positive - mean_negative

# Normalize the vector to have a length of 1
persona_vector = persona_vector / torch.linalg.norm(persona_vector)

print("\nPersona vector calculated successfully.")
print("Vector shape:", persona_vector.shape)



In [ ]:
# =============================================================================
# Cell 5: Save the Persona Vector
# =============================================================================
print(f"Saving vector to {VECTOR_OUTPUT_PATH}...")
os.makedirs(os.path.dirname(VECTOR_OUTPUT_PATH), exist_ok=True)
torch.save(persona_vector, VECTOR_OUTPUT_PATH)
print("Vector saved. This notebook is complete.")